# 02 · Text representations and CPU sentiment training

TA Units 1, 2 and 4. Demonstrate one-hot encoding, BoW, n-grams and TF-IDF, then train a supervised IMDb classifier with a held-out test set. Negation is retained. Tune only against validation data.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from CineScope/notebooks or CineScope.'
sys.path.insert(0, str(ROOT / 'backend'))
sys.path.insert(0, str(ROOT / 'scripts'))
os.chdir(ROOT)
(ROOT / 'artifacts').mkdir(exist_ok=True)
SEED = 42

In [2]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
texts = ['The acting is good but the story is not good.', 'The visuals are stunning and the pacing is slow.', 'The story is excellent and the music is beautiful.']
bow = CountVectorizer(ngram_range=(1,2))
counts = bow.fit_transform(texts)
pd.DataFrame(counts.toarray(),columns=bow.get_feature_names_out())

   acting  acting is  and  ...  the visuals  visuals  visuals are
0       1          1    0  ...            0        0            0
1       0          0    1  ...            1        1            1
2       0          0    1  ...            0        0            0

[3 rows x 38 columns]


In [3]:
vocabulary = ['acting','story','visuals','music']
one_hot = np.eye(len(vocabulary),dtype=int)
display(pd.DataFrame(one_hot,index=vocabulary,columns=vocabulary))
tfidf = TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True)
features = tfidf.fit_transform(texts)
print('Sparse TF-IDF shape:', features.shape)
print('Negation bigram retained:', 'not good' in tfidf.vocabulary_)

         acting  story  visuals  music
acting        1      0        0      0
story         0      1        0      0
visuals       0      0        1      0
music         0      0        0      1
Sparse TF-IDF shape: (3, 38)
Negation bigram retained: True


## Train and evaluate
The function below is implemented in `backend/cinescope/training.py`. It audits duplicates and suspicious patterns, removes exact train/test overlap, respects the official test split, groups validation by movie when IMDb IDs are available, fits vocabulary/IDF on training text only, selects regularization on validation macro-F1, then evaluates test once.

High screening risk is a manual audit flag; it is not a fake-review label. Do not automatically delete all enthusiastic or negative reviews.

In [4]:
RUN_TRAINING = False  # Requires notebook 01's prepared real IMDb CSV.
if RUN_TRAINING:
    from cinescope.training import train_tfidf
    model, report = train_tfidf(ROOT/'data/processed/imdb_reviews.csv',ROOT/'models',ROOT/'artifacts')
    display(report)
else:
    print('Training not run. Set RUN_TRAINING=True after preparing the real IMDb corpus.')

Training not run. Set RUN_TRAINING=True after preparing the real IMDb corpus.


In [5]:
from cinescope.reviews import ReviewAnalyzer
analyzer = ReviewAnalyzer()
for text in ['The story is good.', 'The story is not good.', 'The acting is excellent, but the ending is awful.']:
    print(text, analyzer.score(text))
print('Active model:', analyzer.model_name)

The story is good. {'label': 'positive', 'sentiment_index': 91.7, 'positive_probability': 0.9168, 'confidence': 0.9168, 'uncertain': False, 'method': 'tfidf-logistic-regression'}
The story is not good. {'label': 'negative', 'sentiment_index': 26.5, 'positive_probability': 0.2645, 'confidence': 0.7355, 'uncertain': False, 'method': 'tfidf-logistic-regression'}
The acting is excellent, but the ending is awful. {'label': 'negative', 'sentiment_index': 32.2, 'positive_probability': 0.3225, 'confidence': 0.6775, 'uncertain': False, 'method': 'tfidf-logistic-regression'}
Active model: tfidf-logistic-regression


## Report
Compare unigram vs unigram+bigram TF-IDF with the **same** train/validation split. Report accuracy, macro precision/recall/F1, confusion matrix, ROC-AUC, latency and peak memory. Include a manually inspected error slice for sarcasm, negation, mixed sentiment and long reviews. Raw classifier probabilities are not validated confidence estimates; notebook 07 includes calibration measurement.